## Filter Primer
`Filter` is an object that intercepts HTTP requests before they reach the target `Servlet` and intercepts HTTP responses before they travel back to the client. The interface looks like:

In [ ]:
public interface Filter {
    default void init(FilterConfig filterConfig) throws ServletException {
    }

    void doFilter(ServletRequest request, ServletResponse response, FilterChain chain)
            throws IOException, ServletException;

    default void destroy() {
    }
}

`FilterChain` represents object provided by the servlet container containing an ordered list of matching filters. Calling `chain.doFilter(request, response)` delegates control to the next filter in the chain or to the final servlet if no filters remain.  
![FilterChain](./images/filterchain.png)  

Example `Filter`:

In [ ]:
@WebFilter("/*")
public class LoggingFilter implements Filter {

    @Override
    public void doFilter(ServletRequest request, ServletResponse response, FilterChain chain)
            throws IOException, ServletException {
        
        // 1. Incoming Request Phase
        long startTime = System.currentTimeMillis();
        HttpServletRequest req = (HttpServletRequest) request;
        System.out.println("-> [PRE] Incoming request: " + req.getRequestURI());

        // 2. Pass Control Down the Chain
        // If you omit this call, the request stops here (blocked/short-circuited).
        chain.doFilter(request, response);

        // 3. Outgoing Response Phase (Executes on the way back)
        long duration = System.currentTimeMillis() - startTime;
        System.out.println("<- [POST] Completed processing in " + duration + " ms");
    }
}

### Spring Integration
In traditional Spring application (without Spring Boot), we register `Filter`s as:

In [ ]:
/*
Notice that this class is not a bean. The servlet container recognises this class
and invokes it even before the ApplicationContext has loaded. This is `ServiceLoader` 
SPI mechanism.
*/
public class MyWebAppInitializer implements WebApplicationInitializer {

    @Override
    public void onStartup(ServletContext servletContext) throws ServletException {
        // Filters registered in order of declaration here
        TraceIdFilter filter = new TraceIdFilter();
        var registration = servletContext.addFilter("traceIdFilter", filter);
        registration.addMappingForUrlPatterns(null, false, "/*");

        // ... other Filters
    }
}

/*
Notice that we are calling new TraceIdFilter(); this means that TraceIdFilter
is not registered as a bean. So how do we inject dependencies into it? The answer
lies in the Filter's init() method which executes after ApplicationContext has
loaded.
*/

**DelegatingFilterProxy:** is a thin `Filter` implementation whose job is to look up a Spring beans by name from the `WebApplicationContext` and delegate all `doFilter()` calls to it. As we saw earlier, `TraceIdFilter` was not registered as a Spring bean and thus it had no direct way to autowire components into it. `DelegatingFilterProxy` solves this by acting as a bridge between the servlet container lifecycle (Tomcat, Jetty) and the Spring `ApplicationContext` lifecycle. Here is the flow:
1. Tomcat invokes `DelegatingFilterProxy.doFilter()`.
2. `DelegatingFilterProxy` looks up the real filter bean inside Spring’s `ApplicationContext` by bean name.
3. It delegates the `doFilter()` call to the Spring-managed bean.

In [ ]:
@Component("customAuditFilter")
public class CustomAuditFilter extends Filter {
    // ...
}

public class WebAppInitializer implements WebApplicationInitializer {
    @Override
    public void onStartup(ServletContext servletContext) throws ServletException {
        // Register proxy with Servlet Container, delegating to "customAuditFilter" Spring bean
        var registration = servletContext.addFilter("customAuditFilter", new DelegatingFilterProxy("customAuditFilter"));
        registration.addMappingForUrlPatterns(null, false, "/api/*");
    }
}

Spring Boot changes how filters are registered. Filters go from "servlet-container artifacts that Spring reaches into" to "first-class Spring beans that Boot registers into an embedded servlet container it manages". This is possible because in Spring Boot the servlet container is itself created and managed by Spring.

To register a `Filter` and associate it with all paths (/*), we can just mark the `Filter` as a bean:

In [ ]:
@Component
public class LoggingFilter implements Filter {
    // ...
}

For more control,

In [ ]:
@Configuration
public class FilterConfig {

    @Bean
    public FilterRegistrationBean<LoggingFilter> loggingFilter(LoggingFilter filter) {
        FilterRegistrationBean<LoggingFilter> registration = new FilterRegistrationBean<>();
        registration.setFilter(filter);
        registration.addUrlPatterns("/api/*");
        registration.setDispatcherTypes(DispatcherType.REQUEST, DispatcherType.ASYNC);
        registration.setOrder(1); // lower = earlier in chain
        return registration;
    }
}

/*
We would think that LoggingFilter would get registerd twice here, once through FilterRegistrationBean
mechanism and other through @Component. But this is not the case here.
*/

**OncePerRequestFilter:** to understand this, we need to understand different dispatcher types:
- `DispatcherType.FORWARD`: an internal forward happens entirely inside the server without telling the browser. A servlet (or `Controller`) processes part of a request and passes handling to another internal resource (like a JSP, an error view, or another controller) using `request.getRequestDispatcher("/internal-path").forward(req, res)`.  
  ![Type Forward](./images/dispatcher_type_forward.png)

- `DispatcherType.ASYNC`: this mode works in two phases:
    - Container thread A accepts the request, runs the `FilterChain`, and enters the `Controller`. The `Controller` starts background work and surrenders thread A back to Tomcat's pool.
    - When background work finishes, the container picks container thread B from the pool and redispatches the request back into the `FilterChain` to complete the HTTP response.  
      ![Type Async](./images/dispatcher_type_async.png)

In both of the above cases we see that same `Filter` can get invoked more than once. `OncePerRequestFilter` prevents this:

In [ ]:
// To create a Filter with semantics of OncePerRequestFilter, extend it and override
// doFilterInternal method (not doFilter)

@Component
public class CachingRequestBodyFilter extends OncePerRequestFilter {

    @Override
    protected void doFilterInternal(HttpServletRequest request, 
                                     HttpServletResponse response, 
                                     FilterChain filterChain) throws ServletException, IOException {
        ContentCachingRequestWrapper wrappedRequest = new ContentCachingRequestWrapper(request);
        filterChain.doFilter(wrappedRequest, response);
        // after chain completes, wrappedRequest.getContentAsByteArray() has the cached body
    }
}

## Spring Security Architecture
![Spring Security Architecture](./images/spring_security_arch.png)

**FilterChainProxy:** is the heart of Spring Security. It is a `Filter` that allows delegating to many `Filter` instances through many `SecurityFilterChain`. It is a Spring bean and is responsible for selecting the right `SecurityFilterChain` to use (based on request object).

**SecurityFilterChain:** contains a list of `Filter`s and is used by `FilterChainProxy` to determine which Spring Security filter instances should be invoked for the current request.

**Security Filter:** are also Spring beans but are registered with `FilterChainProxy` instead of `DelegatingFilterProxy`.

Example code that registers a new `SecurityFilterChain`:

In [ ]:
@Configuration
@EnableWebSecurity
public class SecurityConfig {

    @Bean
    public SecurityFilterChain filterChain(HttpSecurity http) throws Exception {
        http
            // Enables Cross-Site Request Forgery (CSRF) protection using standard defaults. 
            // Any state-changing request (POST, PUT, DELETE, PATCH) must include a valid 
            // CSRF token in the request headers or form data. Enabled through CsrfFilter
            .csrf(Customizer.withDefaults())
            // Enables HTTP Basic Authentication. API clients can authenticate by sending 
            // an Authorization: Basic <base64-credentials> header. Enabled through BasicAuthenticationFilter
            .httpBasic(Customizer.withDefaults())
            // Enables Form-Based Login. Browser users attempting to access protected resources
            // without credentials will be redirected to Spring Security's built-in login page at /login.
            // Through UsernamePasswordAuthenticationFilter.
            .formLogin(Customizer.withDefaults())
            // Defines the authorization rule: Every single HTTP request (anyRequest()) must be sent by an 
            // authenticated user. Unauthenticated requests are rejected with a 401 Unauthorized (for HTTP Basic)
            // or redirected to /login (for browser sessions). Through AuthorizationFilter
            .authorizeHttpRequests(authorize -> authorize
                .anyRequest().authenticated()
            )
            // Add any customer filter at any position
            .addFilterAfter(new SecurityAuditFilter(), AuthorizationFilter.class);

        return http.build();
    }

}

/*
Notice again that we wrote new SecurityAuditFilter() and not added @Component to SecurityAuditFilter. This is to
prevent double registration and invocation of filters - once by the servlet container and other by Spring Security
inside SecurityFilterChain. However, if we want to keep the Filter as a bean, we can do the following:

@Bean
public FilterRegistrationBean<SecurityAuditFilter> securityAuditFilterRegistration(SecurityAuditFilter filter) {
    FilterRegistrationBean<SecurityAuditFilter> registration = new FilterRegistrationBean<>(filter);
    registration.setEnabled(false);  // -> key part
    return registration;
}
*/